# 00 — Final clean and merge of CrowdBOT predictions

Produces one validated clean CSV for every model × approach × history configuration: 3 models × 2 approaches × 16 histories = 96 files.

The final policy matches the HuRoN/PAL workflow:

- `predicted_label` is the authoritative explicit class label.
- Original valid probabilities are retained unchanged.
- InternVL formatting failures are recovered only when both an explicit label and a complete valid probability vector can be extracted from the stored raw response.
- No probabilities are fabricated or imputed. Any unrecoverable row stops the notebook.
- The five selectively rerun InternVL AppOd H30 frames must now pass as original valid rows.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

ANALYSIS = Path.cwd().parent if Path.cwd().name == "Codes" else Path.cwd()
RAW = ANALYSIS / "Model_Pred" / "CrowdBot"
CLEAN = ANALYSIS / "Outputs" / "CrowdBot" / "clean"

MODELS = ["qwen", "llava", "internvl"]
APPROACHES = ["appfr", "appod"]
HISTORIES = range(2, 33, 2)
LABELS = ["safe", "potentially_unsafe", "unsafe"]
PROB_COLS = ["prob_safe", "prob_potentially_unsafe", "prob_unsafe"]
PROB_TO_LABEL = dict(zip(PROB_COLS, LABELS))

EXPECTED_BAGS = 7
EXPECTED_FILES = 3 * 2 * 16 * 7
EXPECTED_CONFIGS = 3 * 2 * 16

ALIASES = {
    "bag_name": ["bag_id", "bag_name"],
    "frame_id": ["frame_uid", "frame_id"],
    "frame_index": ["frame_index"],
    "frame_time_s": ["frame_time_s"],
    "frame_rel_time_s": ["frame_rel_time_s"],
    "ground_truth": ["gt_state", "ground_truth", "ground_truth_label"],
    "gt_reason": ["gt_reason"],
    "front_min_distance_m": ["front_min_distance_m"],
    "warning_required": ["warning_required"],
    "warning_actionable": ["warning_actionable"],
    "unsafe_event_id": ["unsafe_event_id"],
    "next_unsafe_time_s": ["next_unsafe_time_s"],
    "lead_time_to_unsafe_s": ["lead_time_to_unsafe_s"],
    "evidence_quality": ["evidence_quality"],
    "evaluation_eligible": ["evaluation_eligible"],
    "model_saved_label": ["predicted_state", "predicted_label"],
    "prediction_valid_original": ["prediction_valid"],
    "prob_safe": ["p_safe", "prob_safe"],
    "prob_potentially_unsafe": ["p_potentially_unsafe", "prob_potentially_unsafe"],
    "prob_unsafe": ["p_unsafe", "prob_unsafe"],
    "input_mode": ["input_mode"],
    "history_frames": ["history_frames"],
    "model_name": ["model_name"],
}

FINAL_COLS = [
    "bag_name", "frame_id", "frame_index", "frame_time_s", "frame_rel_time_s",
    "ground_truth", "gt_reason", "front_min_distance_m", "warning_required",
    "warning_actionable", "unsafe_event_id", "next_unsafe_time_s",
    "lead_time_to_unsafe_s", "evidence_quality", "evaluation_eligible",
    "model_saved_label", "predicted_label", "label_source",
    "prediction_valid_original", "prediction_recovered",
    "repair_status", "probability_argmax_label", "probability_label_disagreement",
    *PROB_COLS, "msp", "pcs", "entropy", "normalized_entropy", "deep_gini",
    "correct_binary",
]

print("Raw:", RAW)
print("Clean:", CLEAN)
assert RAW.is_dir(), f"Missing CrowdBOT raw folder: {RAW}"



In [ ]:
def clean_labels(series):
    return series.astype("string").str.strip().str.lower().replace({
        "potentially unsafe": "potentially_unsafe",
        "potentially-unsafe": "potentially_unsafe",
        "potentiallyunsafe": "potentially_unsafe",
    })


def to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().map({
        "true": True, "false": False, "1": True, "0": False,
    }).fillna(False).astype(bool)


def parse_internvl(value):
    """Return the explicit raw label and unchanged probability values."""
    text = "" if pd.isna(value) else str(value).translate(str.maketrans({
        "“": '"', "”": '"', "‘": "'", "’": "'",
    }))
    number = r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"

    def match(pattern, cast=str):
        found = re.search(pattern, text, flags=re.I)
        return cast(found.group(1)) if found else (pd.NA if cast is str else np.nan)

    return pd.Series({
        "raw_label": match(
            r'["\']?label["\']?\s*:\s*["\']?(safe|potentially_unsafe|unsafe)'
        ),
        "raw_p_safe": match(r'["\']?p_safe["\']?\s*:\s*' + number, float),
        "raw_p_pus": match(
            r'["\']?p_potential(?:ly)?_unsafe["\']?\s*:\s*' + number, float
        ),
        "raw_p_unsafe": match(r'["\']?p_unsafe["\']?\s*:\s*' + number, float),
    })


def read_prediction(file):
    raw = pd.read_csv(file)
    rename = {}
    path_values = {
        "model_name": file.parents[2].name,
        "input_mode": file.parents[1].name,
        "history_frames": int(file.parent.name.removeprefix("h")),
    }
    for target, choices in ALIASES.items():
        source = next((column for column in choices if column in raw.columns), None)
        if source is None and target not in path_values:
            raise ValueError(f"{file}: missing {target}; columns={list(raw.columns)}")
        if source is not None:
            rename[source] = target

    df = raw[list(rename)].rename(columns=rename)
    for target, value in path_values.items():
        if target not in df:
            df[target] = value
    df["model_raw_output"] = raw.get(
        "internvl_raw_output", pd.Series(pd.NA, index=raw.index)
    )
    df["source_file"] = file.name
    return df


def valid_probabilities(values):
    return (
        np.isfinite(values).all(axis=1)
        & (values >= 0).all(axis=1)
        & (values <= 1).all(axis=1)
        & np.isclose(values.sum(axis=1), 1.0, atol=1e-5)
    )


def prepare(df):
    df = df.copy()
    df["ground_truth"] = clean_labels(df["ground_truth"])
    df["model_saved_label"] = clean_labels(df["model_saved_label"])
    df["prediction_valid_original"] = to_bool(df["prediction_valid_original"])
    df[PROB_COLS] = df[PROB_COLS].apply(pd.to_numeric, errors="coerce")

    df["predicted_label"] = df["model_saved_label"]
    df["label_source"] = "saved_prediction"
    df["prediction_recovered"] = False
    df["repair_status"] = "original_valid"

    probs_ok = valid_probabilities(df[PROB_COLS].to_numpy(float))
    label_ok = df["model_saved_label"].isin(LABELS)
    internvl = df["model_name"].astype(str).str.lower().eq("internvl")
    needs_repair = internvl & (~df["prediction_valid_original"] | ~probs_ok | ~label_ok)

    if needs_repair.any():
        recovered = df.loc[needs_repair, "model_raw_output"].apply(parse_internvl)
        raw_prob_cols = ["raw_p_safe", "raw_p_pus", "raw_p_unsafe"]
        raw_label_ok = recovered["raw_label"].isin(LABELS)
        raw_probs_ok = valid_probabilities(recovered[raw_prob_cols].to_numpy(float))
        recoverable = raw_label_ok & raw_probs_ok
        exact = recovered.index[recoverable]
        failed = recovered.index[~recoverable]

        if len(failed):
            audit = df.loc[failed, [
                "source_file", "bag_name", "frame_id",
                "prediction_valid_original", "model_saved_label",
            ]].copy()
            audit["raw_label"] = recovered.loc[failed, "raw_label"]
            audit["raw_p_safe"] = recovered.loc[failed, "raw_p_safe"]
            audit["raw_p_pus"] = recovered.loc[failed, "raw_p_pus"]
            audit["raw_p_unsafe"] = recovered.loc[failed, "raw_p_unsafe"]
            raise ValueError(
                "Unrecoverable InternVL rows remain. "
                "Do not impute probabilities; rerun these frames:\n"
                + audit.to_string(index=False)
            )

        if len(exact):
            df.loc[exact, "predicted_label"] = recovered.loc[exact, "raw_label"]
            df.loc[exact, PROB_COLS] = recovered.loc[exact, raw_prob_cols].to_numpy()
            df.loc[exact, ["label_source", "prediction_recovered", "repair_status"]] = [
                "raw_output", True, "exact_raw_recovery"
            ]

    probabilities = df[PROB_COLS].to_numpy(float)
    if not valid_probabilities(probabilities).all():
        raise ValueError("Invalid probabilities remain after recovery")
    if not df["predicted_label"].isin(LABELS).all():
        raise ValueError("Invalid prediction labels remain after recovery")

    df["probability_argmax_label"] = df[PROB_COLS].idxmax(axis=1).map(PROB_TO_LABEL)
    df["probability_label_disagreement"] = (
        df["predicted_label"] != df["probability_argmax_label"]
    )
    df["correct_binary"] = (df["ground_truth"] == df["predicted_label"]).astype(int)
    df["msp"] = probabilities.max(axis=1)
    ordered = np.sort(probabilities, axis=1)
    df["pcs"] = ordered[:, -1] - ordered[:, -2]
    df["entropy"] = -(probabilities * np.log(probabilities + 1e-12)).sum(axis=1)
    df["normalized_entropy"] = df["entropy"] / np.log(len(LABELS))
    df["deep_gini"] = 1 - np.square(probabilities).sum(axis=1)
    return df



In [ ]:
# Discover the complete experimental grid.
groups = {}
for model in MODELS:
    for approach in APPROACHES:
        for history in HISTORIES:
            folder = RAW / model / approach / f"h{history:02d}"
            assert folder.is_dir(), f"Missing folder: {folder}"
            files = sorted(folder.glob("*.csv"))
            assert len(files) == EXPECTED_BAGS, (
                f"{model} {approach} H{history:02d}: expected 7 files, found {len(files)}"
            )
            groups[(model, approach, history)] = files

assert sum(map(len, groups.values())) == EXPECTED_FILES
print(f"Discovered {EXPECTED_FILES} raw files in {EXPECTED_CONFIGS} configurations.")

# Qwen AppFr H02 defines the canonical frame population.
canonical = pd.concat(
    [read_prediction(file) for file in groups[("qwen", "appfr", 2)]],
    ignore_index=True,
)
canonical["ground_truth"] = clean_labels(canonical["ground_truth"])
CANONICAL_ROWS = len(canonical)
canonical_identity = canonical[
    ["bag_name", "frame_id", "frame_time_s", "ground_truth"]
].sort_values(["bag_name", "frame_id"]).reset_index(drop=True)

assert CANONICAL_ROWS == 8801, f"Expected 8,801 canonical rows, found {CANONICAL_ROWS:,}"
assert canonical["bag_name"].nunique() == EXPECTED_BAGS
assert set(canonical["ground_truth"].dropna()) <= set(LABELS)
assert not canonical.duplicated(["bag_name", "frame_id"]).any()

gt_by_bag = pd.crosstab(canonical["bag_name"], canonical["ground_truth"]).reindex(
    columns=LABELS, fill_value=0
)
gt_by_bag.loc["TOTAL"] = gt_by_bag.sum()
display(gt_by_bag)
print("Canonical rows:", CANONICAL_ROWS)



In [ ]:
# Validate, merge, and write all configurations.
CLEAN.mkdir(parents=True, exist_ok=True)
summary_rows, repair_rows = [], []

for model in MODELS:
    (CLEAN / model).mkdir(parents=True, exist_ok=True)
    for approach in APPROACHES:
        for history in HISTORIES:
            files = groups[(model, approach, history)]
            df = prepare(pd.concat([read_prediction(file) for file in files], ignore_index=True))
            df = df.sort_values(["bag_name", "frame_id"]).reset_index(drop=True)
            name = f"{model} {approach} H{history:02d}"

            assert len(df) == CANONICAL_ROWS, f"{name}: wrong row count"
            assert df["bag_name"].nunique() == EXPECTED_BAGS, f"{name}: wrong bag count"
            assert not df.duplicated(["bag_name", "frame_id"]).any(), f"{name}: duplicates"
            identity = df[["bag_name", "frame_id", "frame_time_s", "ground_truth"]].reset_index(drop=True)
            assert identity[["bag_name", "frame_id", "ground_truth"]].equals(
                canonical_identity[["bag_name", "frame_id", "ground_truth"]]
            ), f"{name}: frame/GT mismatch"
            assert np.allclose(
                pd.to_numeric(identity["frame_time_s"]),
                pd.to_numeric(canonical_identity["frame_time_s"]),
                atol=1e-6,
                rtol=0,
            ), f"{name}: timestamp mismatch"

            output = CLEAN / model / f"{model}_{approach}_h{history:02d}.csv"
            df[FINAL_COLS].to_csv(output, index=False)

            repaired = df.loc[df["prediction_recovered"], [
                "source_file", "bag_name", "frame_id", "model_saved_label",
                "predicted_label", "repair_status", *PROB_COLS,
            ]].copy()
            if not repaired.empty:
                repaired.insert(0, "history", f"H{history:02d}")
                repaired.insert(0, "approach", approach)
                repaired.insert(0, "model", model)
                repair_rows.append(repaired)

            summary_rows.append({
                "model": model,
                "approach": approach,
                "history": f"H{history:02d}",
                "source_files": len(files),
                "bags": df["bag_name"].nunique(),
                "rows": len(df),
                "recovered_rows": int(df["prediction_recovered"].sum()),
                "unrecoverable_rows": 0,
                "label_probability_disagreements": int(
                    df["probability_label_disagreement"].sum()
                ),
                "accuracy_check": df["correct_binary"].mean(),
                "output_file": output.name,
            })
            print("Saved", name)

summary = pd.DataFrame(summary_rows)
repairs = pd.concat(repair_rows, ignore_index=True) if repair_rows else pd.DataFrame()
summary.to_csv(CLEAN / "merge_summary.csv", index=False)
repairs.to_csv(CLEAN / "repair_audit.csv", index=False)
gt_by_bag.to_csv(CLEAN / "crowdbot_ground_truth_counts.csv")

assert len(summary) == EXPECTED_CONFIGS
display(summary.groupby("model").agg(
    clean_files=("output_file", "count"),
    rows_per_file=("rows", "first"),
    recovered_rows=("recovered_rows", "sum"),
    unrecoverable_rows=("unrecoverable_rows", "sum"),
))

# Final frozen-data expectations after syncing the five-frame EX3 rerun.
assert summary["rows"].eq(8801).all()
assert summary["unrecoverable_rows"].eq(0).all()
assert int(summary.loc[summary["model"].eq("internvl"), "recovered_rows"].sum()) == 18632
assert int(summary.loc[
    summary["model"].eq("internvl")
    & summary["approach"].eq("appod")
    & summary["history"].eq("H30"),
    "recovered_rows",
].iloc[0]) == 7272

print("Final recovery expectations passed: 18,632 exact InternVL recoveries; 0 imputed rows.")



In [ ]:
# Reload every written file and verify the final schema.
expected_outputs = [
    CLEAN / model / f"{model}_{approach}_h{history:02d}.csv"
    for model in MODELS
    for approach in APPROACHES
    for history in HISTORIES
]

for output in expected_outputs:
    df = pd.read_csv(output)
    assert len(df) == CANONICAL_ROWS, f"{output.name}: wrong row count"
    assert list(df.columns) == FINAL_COLS, f"{output.name}: wrong columns"
    assert not df.duplicated(["bag_name", "frame_id"]).any(), f"{output.name}: duplicates"
    assert df["predicted_label"].isin(LABELS).all(), f"{output.name}: invalid labels"
    assert df[PROB_COLS].notna().all().all(), f"{output.name}: missing probabilities"
    assert valid_probabilities(df[PROB_COLS].to_numpy(float)).all(), (
        f"{output.name}: invalid probabilities"
    )
    assert not df["repair_status"].eq("provisional_one_hot").any(), (
        f"{output.name}: provisional probability rows remain"
    )

print(f"Final verification passed: {len(expected_outputs)} clean files in {CLEAN}")


In [ ]:
approach_summary = (
    summary
    .groupby(
        ["model", "approach"],
        as_index=False,
    )
    .agg(
        clean_files=("output_file", "count"),
        rows_per_file=("rows", "first"),
        source_files=("source_files", "sum"),
        recovered_rows=("recovered_rows", "sum"),
        unrecoverable_rows=("unrecoverable_rows", "sum"),
        label_probability_disagreements=(
            "label_probability_disagreements",
            "sum",
        ),
    )
    .sort_values(["model", "approach"])
    .reset_index(drop=True)
)

assert len(approach_summary) == 6
assert approach_summary["clean_files"].eq(16).all()
assert approach_summary["rows_per_file"].eq(8801).all()
assert approach_summary["source_files"].eq(112).all()
assert approach_summary["unrecoverable_rows"].eq(0).all()

display(approach_summary)

approach_summary.to_csv(
    CLEAN / "merge_summary_by_model_approach.csv",
    index=False,
)

print(
    "Saved:",
    CLEAN / "merge_summary_by_model_approach.csv",
)